# 04b — Analisis Spasial (Fase 4B · RQ2, RQ4)

**Pertanyaan:** zona mana yang menjadi hotspot / cold spot pickup, dan apakah stabil antarbulan?

**Keputusan metode** (catat juga di `docs/analysis_plan.md`):

| Item | Pilihan | Alasan |
| :-- | :-- | :-- |
| Variabel | `log1p(total pickup per zona)` | Distribusi pickup sangat miring; tanpa log, Manhattan mendominasi z-score |
| Spatial weights | KNN, k = 8, centroid poligon (EPSG:2263) | Setiap zona pasti punya tetangga, jadi tidak ada zona *island* yang perlu penanganan khusus |
| Global | Moran's I (row-standardized, 999 permutasi) | |
| Lokal | Getis-Ord Gi\* (binary weights, 999 permutasi) | |
| Koreksi uji berganda | FDR Benjamini-Hochberg, α = 0,05, pada p-value dua sisi (2 × `p_sim`) | |
| Kelas | hotspot: z > 0 & signifikan · coldspot: z < 0 & signifikan | |
| Stabilitas | Jaccard Index antarbulan, dihitung terpisah untuk hotspot dan coldspot | |

**Masukan:** `datasets/interim/2026/demand_zone_hour.parquet` (dari Anggota 2) dengan kolom `hour`, `PU_longitude`, `PU_latitude`, `pickup_count`. Jika nama file/kolom berbeda, ubah hanya sel konfigurasi di bawah.

In [11]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from libpysal.weights import KNN
from esda import Moran, G_Local
from statsmodels.stats.multitest import multipletests

SEED = 42            # seed global, lihat docs/analysis_plan.md
K = 8
ALPHA = 0.05
PERMS = 999

ROOT = Path("..")
DEMAND = ROOT / "datasets/interim/2026/demand_zone_hour.parquet"
COL_HOUR, COL_LON, COL_LAT, COL_COUNT = "hour", "PU_longitude", "PU_latitude", "pickup_count"
OUT_CSV = ROOT / "out/output_eda"
OUT_HTML = ROOT / "out/html"
OUT_CSV.mkdir(parents=True, exist_ok=True)
OUT_HTML.mkdir(parents=True, exist_ok=True)

## 1. Zona dan spatial weights

In [12]:
zones = gpd.read_file(ROOT / "datasets/zone/taxi_zones/taxi_zones.shp")   # EPSG:2263
ref = pd.read_parquet(ROOT / "datasets/zone/taxi_zone.parquet")[["LocationID", "longitude", "latitude"]]
zones = zones.merge(ref, on="LocationID", how="left").sort_values("LocationID").reset_index(drop=True)
assert zones[["longitude", "latitude"]].notna().all().all()
assert zones["LocationID"].is_unique

cent = zones.geometry.centroid
w = KNN.from_array(np.c_[cent.x, cent.y], k=K)
print(f"{len(zones)} zona | k = {K} | zona tanpa tetangga: {len(w.islands)}")

263 zona | k = 8 | zona tanpa tetangga: 0


## 2. Pickup per zona per bulan

In [13]:
d = pd.read_parquet(DEMAND, columns=[COL_HOUR, COL_LON, COL_LAT, COL_COUNT])
d["month"] = pd.to_datetime(d[COL_HOUR]).dt.month

# koordinat adalah centroid zona -> kembalikan ke LocationID lewat tabel pemetaan
key = ref.assign(lon_k=ref["longitude"].round(6), lat_k=ref["latitude"].round(6))[["LocationID", "lon_k", "lat_k"]]
d["lon_k"], d["lat_k"] = d[COL_LON].round(6), d[COL_LAT].round(6)
d = d.merge(key, on=["lon_k", "lat_k"], how="left")

unmatched = d["LocationID"].isna().sum()
print(f"baris tanpa pasangan zona: {unmatched} ({unmatched / len(d):.4%})")
assert unmatched == 0, "Koordinat tidak cocok dengan taxi_zone.parquet; periksa tabel pemetaan"

monthly = (d.groupby(["LocationID", "month"])[COL_COUNT].sum().unstack(fill_value=0)
             .reindex(zones["LocationID"], fill_value=0).fillna(0))
monthly["all"] = monthly.sum(axis=1)
assert monthly["all"].sum() == d[COL_COUNT].sum()    # total pickup agregat = sumber
months = [m for m in monthly.columns if m != "all"]
monthly.sum().rename("total_pickup").to_frame()

baris tanpa pasangan zona: 0 (0.0000%)


,total_pickup
month,
1,3615518.0
2,3310826.0
3,3858721.0
4,3743693.0
5,3999128.0
6,3746664.0
all,22274550.0


## 3. Moran's I dan Gi\* (fungsi)

In [14]:
def global_moran(y):
    np.random.seed(SEED)
    m = Moran(y, w, transformation="R", permutations=PERMS)
    return {"I": m.I, "E[I]": m.EI, "z_sim": m.z_sim, "p_sim": m.p_sim}

def gi_star(y):
    g = G_Local(y, w, transform="B", star=True, permutations=PERMS, seed=SEED)
    p = np.minimum(g.p_sim * 2, 1)                                 # p_sim satu sisi -> dua sisi
    sig, q, _, _ = multipletests(p, alpha=ALPHA, method="fdr_bh")
    kelas = np.where(sig & (g.Zs > 0), "hotspot", np.where(sig & (g.Zs < 0), "coldspot", "ns"))
    return pd.DataFrame({"z": g.Zs, "p": p, "q_fdr": q, "kelas": kelas}, index=zones["LocationID"])

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else np.nan

## 4. Keseluruhan Januari–Juni

In [15]:
y_all = np.log1p(monthly["all"].to_numpy())
moran_all = global_moran(y_all)
gi_all = gi_star(y_all)
print({k: round(float(v), 4) for k, v in moran_all.items()})
gi_all["kelas"].value_counts().rename("jumlah_zona").to_frame()

{'I': 0.6937, 'E[I]': -0.0038, 'z_sim': 23.4177, 'p_sim': 0.001}


,jumlah_zona
kelas,
ns,187
hotspot,55
coldspot,21


In [16]:
res = zones[["LocationID", "zone", "borough", "geometry"]].merge(gi_all.reset_index(), on="LocationID")
res.to_crs(4326).explore(
    column="kelas", categorical=True,
    categories=["hotspot", "coldspot", "ns"], cmap=["#d73027", "#4575b4", "#d9d9d9"],
    tooltip=["zone", "borough", "kelas", "z", "q_fdr"], tiles="OpenStreetMap",
).save(OUT_HTML / "04b_hotspot_all.html")
res.drop(columns="geometry").to_csv(OUT_CSV / "04b_gistar_zone_all.csv", index=False)
print("peta:", OUT_HTML / "04b_hotspot_all.html")

peta: ..\out\html\04b_hotspot_all.html


## 5. Per bulan dan stabilitas (Jaccard)

In [17]:
moran_rows, kelas_bulan = [], {}
for m in months:
    y = np.log1p(monthly[m].to_numpy())
    moran_rows.append({"bulan": m, **global_moran(y)})
    kelas_bulan[m] = gi_star(y)["kelas"]
moran_tbl = pd.DataFrame([{"bulan": "Jan–Jun", **moran_all}] + moran_rows)
kelas_bulan = pd.DataFrame(kelas_bulan)
moran_tbl.to_csv(OUT_CSV / "04b_moran_global.csv", index=False)
kelas_bulan.to_csv(OUT_CSV / "04b_kelas_per_bulan.csv")
moran_tbl

,bulan,I,E[I],z_sim,p_sim
0,Jan–Jun,0.693713,-0.003817,23.417737,0.001
1,1,0.692767,-0.003817,23.387128,0.001
2,2,0.696125,-0.003817,23.377569,0.001
3,3,0.686291,-0.003817,23.061482,0.001
4,4,0.704694,-0.003817,23.785763,0.001
5,5,0.729366,-0.003817,24.584201,0.001
6,6,0.758064,-0.003817,25.800910,0.001


In [18]:
def jaccard_matrix(label):
    ids = {m: kelas_bulan.index[kelas_bulan[m] == label] for m in months}
    return pd.DataFrame([[jaccard(ids[a], ids[b]) for b in months] for a in months], index=months, columns=months)

J_hot, J_cold = jaccard_matrix("hotspot"), jaccard_matrix("coldspot")
J_hot.to_csv(OUT_CSV / "04b_jaccard_hotspot.csv")
J_cold.to_csv(OUT_CSV / "04b_jaccard_coldspot.csv")

jumlah = pd.DataFrame({lab: (kelas_bulan == lab).sum() for lab in ["hotspot", "coldspot"]})
rata2 = lambda J: np.mean([J.loc[a, b] for a, b in zip(months[:-1], months[1:])])
print("Jumlah zona per bulan:\n", jumlah, sep="")
print(f"\nJaccard bulan berurutan (rata-rata): hotspot = {rata2(J_hot):.3f} | coldspot = {rata2(J_cold):.3f}")
J_hot.round(3)

Jumlah zona per bulan:
   hotspot  coldspot
1       50        21
2       50        21
3       55        21
4       55        21
5       54        21
6       56        22

Jaccard bulan berurutan (rata-rata): hotspot = 0.963 | coldspot = 0.991


,1,2,3,4,5,6
1,1.000,0.961,0.909,0.909,0.926,0.893
2,0.961,1.000,0.909,0.909,0.926,0.893
3,0.909,0.909,1.000,1.000,0.982,0.982
4,0.909,0.909,1.000,1.000,0.982,0.982
5,0.926,0.926,0.982,0.982,1.000,0.964
6,0.893,0.893,0.982,0.982,0.964,1.000


## 6. Kesimpulan RQ2 dan masukan RQ4

_Isi setelah notebook dijalankan pada `demand_zone_hour` final:_

- Moran's I (Jan–Jun) dan arti autokorelasinya: …
- Zona hotspot / coldspot utama: …
- Stabilitas antarbulan (rujuk Jaccard di atas): …
- Zona prioritas untuk RQ4: …